In [1]:
# 02_decade_and_volatility_analysis.ipynb

import pandas as pd
import numpy as np

print("Libraries loaded successfully.")

# ------------------------------------------------------------
# 1. Load the cleaned CPI data from notebook 1
# ------------------------------------------------------------

# If this file is in the project root, adjust path as needed
df = pd.read_csv("cleaned_cpi_data_initial.csv")

print("Data loaded.")
print("Shape:", df.shape)
display(df.head(3))

# Ensure required columns exist
required_cols = ["Year", "Percent change", "Consumer Price Index item"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ------------------------------------------------------------
# 2. Create Decade column
# ------------------------------------------------------------

df["Decade"] = (df["Year"] // 10) * 10

print("\nDecade column added.")
display(df[["Year", "Decade"]].head(10))

# ------------------------------------------------------------
# 3. Average inflation by decade
# ------------------------------------------------------------

inflation_by_decade = (
    df.groupby("Decade")["Percent change"]
      .mean()
      .sort_values(ascending=False)
)

print("\nAVERAGE FOOD INFLATION BY DECADE (1974-2024)")
print("=" * 60)
print(inflation_by_decade)

worst_decade = inflation_by_decade.index[0]
worst_value = inflation_by_decade.iloc[0]
best_decade = inflation_by_decade.index[-1]
best_value = inflation_by_decade.iloc[-1]

print(f"\nWorst decade: {worst_decade}s with {worst_value:.2f}% average inflation")
print(f"Best decade:  {best_decade}s with {best_value:.2f}% average inflation")

# ------------------------------------------------------------
# 4. Detailed statistics per decade
# ------------------------------------------------------------

decade_stats = df.groupby("Decade")["Percent change"].agg([
    ("Average", "mean"),
    ("Volatility", "std"),
    ("Minimum", "min"),
    ("Maximum", "max"),
    ("Data_Points", "count"),
]).round(2)

print("\nDETAILED DECADE STATISTICS")
print("=" * 60)
print(decade_stats)

print("\nInterpretation:")
print("- Average = typical inflation for that decade")
print("- Volatility = how much prices swung (higher = more unpredictable)")
print("- Maximum = biggest price spike in that decade")

# ------------------------------------------------------------
# 5. Volatility by food category
# ------------------------------------------------------------

volatility_by_item = (
    df.groupby("Consumer Price Index item")["Percent change"]
      .std()
      .sort_values(ascending=False)
)

print("\nMOST VOLATILE FOODS (TOP 10)")
print("=" * 60)
print(volatility_by_item.head(10))

volatility_all = (
    df.groupby("Consumer Price Index item")["Percent change"]
      .std()
      .sort_values()
)

print("\nMOST STABLE (PREDICTABLE) FOODS (TOP 10)")
print("=" * 60)
print(volatility_all.head(10))

print("\nCOMPARISON: Most volatile vs most stable")
print("=" * 60)
print("\nMost volatile (last 5):")
print(volatility_all.tail(5))
print("\nMost stable (first 5):")
print(volatility_all.head(5))

# ------------------------------------------------------------
# 6. Price drops (negative inflation) by category
# ------------------------------------------------------------

price_drops = df[df["Percent change"] < 0]
drops_by_category = (
    price_drops.groupby("Consumer Price Index item")
               .size()
               .sort_values(ascending=False)
)

print("\nFOOD CATEGORIES WITH MOST PRICE DROPS (1974-2024)")
print("=" * 60)
print(drops_by_category.head(10))

print(f"\nMost deflation-prone: {drops_by_category.index[0]}")
print(f"Had {drops_by_category.iloc[0]} years of price drops.")

# ------------------------------------------------------------
# 7. Save outputs
# ------------------------------------------------------------

df.to_csv("abhinaya_cleaned_food_inflation.csv", index=False)
decade_stats.to_csv("abhinaya_decade_analysis.csv")

summary = {
    "Total_Records": int(len(df)),
    "Food_Categories": int(df["Consumer Price Index item"].nunique()),
    "Years_Covered": f"{int(df['Year'].min())}-{int(df['Year'].max())}",
    "Worst_Decade": f"{worst_decade}s ({worst_value:.2f}% avg)",
    "Best_Decade": f"{best_decade}s ({best_value:.2f}% avg)",
}

import json
with open("abhinaya_project_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved files:")
print("- abhinaya_cleaned_food_inflation.csv")
print("- abhinaya_decade_analysis.csv")
print("- abhinaya_project_summary.json")


Libraries loaded successfully.
Data loaded.
Shape: (1122, 3)


,Consumer Price Index item,Year,Percent change
0,All food,1974,14.3
1,All food,1975,8.5
2,All food,1976,3.0



Decade column added.


,Year,Decade
0,1974,1970
1,1975,1970
2,1976,1970
3,1977,1970
4,1978,1970
5,1979,1970
6,1980,1980
7,1981,1980
8,1982,1980
9,1983,1980



AVERAGE FOOD INFLATION BY DECADE (1974-2024)
Decade
1970    8.794444
2020    5.069091
1980    4.552857
2000    2.981818
1990    2.834123
2010    1.473636
Name: Percent change, dtype: float64

Worst decade: 1970s with 8.79% average inflation
Best decade:  2010s with 1.47% average inflation

DETAILED DECADE STATISTICS
        Average  Volatility  Minimum  Maximum  Data_Points
Decade                                                    
1970       8.79        9.63    -12.5     52.4          126
1980       4.55        4.65    -16.6     26.6          210
1990       2.83        3.46    -10.6     17.9          211
2000       2.98        3.70    -14.7     29.2          220
2010       1.47        3.52    -21.1     17.8          220
2020       5.07        4.65     -1.9     32.2          110

Interpretation:
- Average = typical inflation for that decade
- Volatility = how much prices swung (higher = more unpredictable)
- Maximum = biggest price spike in that decade

MOST VOLATILE FOODS (TOP 10)
Co